<a href="https://colab.research.google.com/github/akameCQ/Glaucoma-Segmentation-MiTB3/blob/main/Glaucoma_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"""Mount Google Drive"""

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Extract the ZIP file
!rm -rf /content/data  # remove the old directory
!unzip -q "/content/drive/MyDrive/Dataset/mix_ref_dris_data.zip" -d /content/data

print("✅ Data loaded successfully!")
print("Total Images:", len(os.listdir("/content/data/images")))

Consolidate YOLO and SOTA datasets from Drive

In [ ]:
from google.colab import drive
import os
import shutil
import zipfile
from tqdm import tqdm

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Completely clean legacy files and temporary directories
!rm -rf /content/data
!rm -rf /content/temp_extract

# Create fresh target directories
FINAL_IMG_DIR = "/content/data/images"
FINAL_MASK_DIR = "/content/data/masks"
os.makedirs(FINAL_IMG_DIR, exist_ok=True)
os.makedirs(FINAL_MASK_DIR, exist_ok=True)

# --- ZIP PATHS FOR RETINAL DATA ---
zip_sources = [
    ("YOLO_SET", "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/YOLO.zip"),
    ("SOTA_SET", "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/SOTA.zip"),
    ("TEST_SET", "/content/drive/MyDrive/yeni_glokom_proje/test_data/TEST.zip")
]

print("\n📦 INITIALIZING LARGE-SCALE DATA MERGE...\n")

for set_name, zip_path in zip_sources:
    if not os.path.exists(zip_path):
        print(f"⚠️ WARNING: {set_name} not found at specified Drive path, skipping: {zip_path}")
        continue

    print(f"📦 Extracting {set_name} to temporary storage...")
    temp_sub_dir = f"/content/temp_extract/{set_name}"

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(temp_sub_dir)

    # Deep scan extracted files (fixes folder structure issues)
    for root, dirs, files in os.walk(temp_sub_dir):
        folder_name = os.path.basename(root).lower()

        # If folder contains images
        if "image" in folder_name:
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')) and not f.startswith('.'):
                    # Add set_name prefix to avoid filename collisions
                    yeni_isim = f"{set_name}_{f}"
                    shutil.copy2(os.path.join(root, f), os.path.join(FINAL_IMG_DIR, yeni_isim))

        # If folder contains masks (Ground Truth)
        elif "mask" in folder_name:
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')) and not f.startswith('.'):
                    yeni_isim = f"{set_name}_{f}"
                    shutil.copy2(os.path.join(root, f), os.path.join(FINAL_MASK_DIR, yeni_isim))

# Discard temporary extraction folder
!rm -rf /content/temp_extract

print("\n🔍 ================= CHECKPOINT ================= 🔍")
if os.path.exists(FINAL_IMG_DIR):
    print(f"📸 Total Images in Training Pool: {len(os.listdir(FINAL_IMG_DIR))}")
if os.path.exists(FINAL_MASK_DIR):
    print(f"🎭 Total Masks in Training Pool:   {len(os.listdir(FINAL_MASK_DIR))}")
print("🔍 ============================================== 🔍")
print("\n🎉 Phase 1 successfully completed! Proceed to the next cell.")

Library Installations

In [ ]:
!pip install segmentation-models-pytorch timm
import torch
import segmentation_models_pytorch as smp
print("✅ Libraries are ready!")

Model Architecture (3-Channel Mode)

In [ ]:
# Model: Mix Vision Transformer (MiT) based Unet
model = smp.Unet(
    encoder_name="mit_b3",        # Transformer Encoder
    encoder_weights="imagenet",   # Pre-trained initialization
    in_channels=3,
    classes=3,                    # 0: Background, 1: Disc, 2: Cup
).cuda() # Move model to GPU
print("🚀 MiT-B3 Segmentation Model ready on GPU!")

Model Architecture (Updated for Grayscale 1-Channel Mode)

In [ ]:
model = smp.Unet(
    encoder_name="mit_b3",
    encoder_weights="imagenet",
    in_channels=1,
    classes=3,
).cuda()
print("🚀 1-Channel MiT-B3 Model ready on GPU!")

Hybrid Loss and Optimizer

In [ ]:
from torch import nn
# Loss Functions
criterion_ce = nn.CrossEntropyLoss()
criterion_dice = smp.losses.DiceLoss(mode='multiclass')
# Optimizer configuration
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
def calculate_loss(pred, target):
    return 0.5 * criterion_ce(pred, target) + 0.5 * criterion_dice(pred, target)
print("✅ Hybrid Loss system and Optimizer initialized!")

Hybrid Loss and Optimizer (Adjusted Parameters)

In [ ]:
from torch import nn
# Loss Functions
criterion_ce = nn.CrossEntropyLoss()
criterion_dice = smp.losses.DiceLoss(mode='multiclass')
# Optimizer configuration (Increased LR and Weight Decay added)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
def calculate_loss(pred, target):
    return 0.5 * criterion_ce(pred, target) + 0.5 * criterion_dice(pred, target)
print("✅ Hybrid Loss system and Optimizer initialized!")


Hybrid Loss Configuration (20% CE / 80% Dice)

In [ ]:
from torch import nn
# Loss Functions
criterion_ce = nn.CrossEntropyLoss()
criterion_dice = smp.losses.DiceLoss(mode='multiclass')
# Optimizer configuration
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
def calculate_loss(pred, target):
    return 0.2 * criterion_ce(pred, target) + 0.8 * criterion_dice(pred, target)
print("✅ Hybrid Loss system and Optimizer initialized!")

Training Method 2 (Adjusted LR)

In [ ]:
from torch import nn
# Loss Functions
criterion_ce = nn.CrossEntropyLoss()
criterion_dice = smp.losses.DiceLoss(mode='multiclass')
# Optimizer configuration
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
def calculate_loss(pred, target):
    return 0.2 * criterion_ce(pred, target) + 0.8 * criterion_dice(pred, target)
print("✅ Hybrid Loss system and Optimizer initialized!")


Focal + Dice Loss for Fine-Tuning

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class HybridLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, dice_weight=0.5, focal_weight=0.5):
        super(HybridLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
    def forward(self, inputs, targets):
        # 1. Focal Loss (Heavily penalizes hard-to-classify pixels)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        focal_loss = focal_loss.mean()
        # 2. Dice Loss (Maintains structural integrity)
        inputs_soft = F.softmax(inputs, dim=1)
        num_classes = inputs.size(1)
        dice_loss = 0
        for i in range(1, num_classes):
            input_flat = inputs_soft[:, i, :, :].reshape(-1)
            target_flat = (targets == i).float().reshape(-1)
            intersection = (input_flat * target_flat).sum()
            dice = (2. * intersection + 1e-6) / (input_flat.sum() + target_flat.sum() + 1e-6)
            dice_loss += (1 - dice)
        dice_loss /= (num_classes - 1)
        return (self.focal_weight * focal_loss) + (self.dice_weight * dice_loss)
# Activate criterion
criterion = HybridLoss().cuda()


#Pre-Processing & Extraction


In [ ]:
# 1. Clean up legacy directories if they exist
!rm -rf /content/YOLO_Ile_Kirpilacaklar
!rm -rf /content/SOTA_Hazir_Kirpilmislar
!rm -rf /content/temp_yolo_zip
!rm -rf /content/temp_sota_zip
print("📦 Folder 1: Extracting YOLO ZIP to temporary directory...")
!unzip -q "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/YOLO.zip" -d /content/temp_yolo_zip
print("📦 Folder 2: Extracting SOTA ZIP to temporary directory...")
!unzip -q "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/SOTA.zip" -d /content/temp_sota_zip
# 2. Targeted Extraction & Directory Mapping
import shutil
import os
# Route YOLO files
if os.path.exists("/content/temp_yolo_zip/YOLO_Ile_Kirpilacaklar"):
    shutil.move("/content/temp_yolo_zip/YOLO_Ile_Kirpilacaklar", "/content/YOLO_Ile_Kirpilacaklar")
    print("✅ YOLO data successfully positioned!")
else:
    print("⚠️ YOLO folder name mismatch, please verify temporary folder structure.")
# Route SOTA files
if os.path.exists("/content/temp_sota_zip/SOTA_Hazir_Kirpilmislar"):
    shutil.move("/content/temp_sota_zip/SOTA_Hazir_Kirpilmislar", "/content/SOTA_Hazir_Kirpilmislar")
    print("✅ SOTA data successfully positioned!")
else:
    print("⚠️ SOTA folder name mismatch, please verify temporary folder structure.")
# 3. Clean up temporary trash folders
!rm -rf /content/temp_yolo_zip
!rm -rf /content/temp_sota_zip
print("\n🎉 ZIP extraction and positioning complete! Proceed to Pre-Processing.")

Common SOTA Functions & Directory Setup (YOLO Cropping)

In [ ]:
import os, cv2, shutil, numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
# 1. CLEANUP AND SETUP
DRIVE_IMG = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
DRIVE_MASK = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
LOCAL_IMG = "/content/new_train_data/raw_cropped"
LOCAL_MASK = "/content/new_train_data/masks"
# Reset directories
for path in [DRIVE_IMG, DRIVE_MASK, LOCAL_IMG, LOCAL_MASK]:
    if os.path.exists(path): shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
yolo_model = YOLO('/content/drive/MyDrive/yeni_glokom_proje/yolo_save/yolo_disk_dedektor_yeniv2.pt')
IMAGES_DIR = "/content/YOLO_Ile_Kirpilacaklar/images"
MASKS_DIR = "/content/YOLO_Ile_Kirpilacaklar/masks"
files = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"🚀 Scanning {len(files)} images, processing those with valid masks...")
# 2. PROCESSING LOOP
for idx, filename in enumerate(tqdm(files)):
    mask_path = os.path.join(MASKS_DIR, filename)
    if not os.path.exists(mask_path): continue
    img = cv2.imread(os.path.join(IMAGES_DIR, filename))
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: continue
    h, w = img.shape[:2]
    is_refuge = "refuge" in filename.lower() or max(h, w) > 1200
    if is_refuge:
        pts = np.argwhere(mask > 0)
        if len(pts) > 0:
            cy, cx = int(np.mean(pts[:, 0])), int(np.mean(pts[:, 1]))
            y_min, y_max = max(0, cy - 225), min(h, cy + 225)
            x_min, x_max = max(0, cx - 225), min(w, cx + 225)
        else:
            y_min, y_max, x_min, x_max = h//2-250, h//2+250, w//2-250, w//2+250
    else:
        results = yolo_model.predict(img, conf=0.4, verbose=False)
        if len(results[0].boxes) > 0:
            x1, y1, x2, y2 = map(int, results[0].boxes.xyxy[0])
            pad = int((x2 - x1) * 0.15)
            y_min, y_max = max(0, y1 - pad), min(h, y2 + pad)
            x_min, x_max = max(0, x1 - pad), min(w, x2 + pad)
        else:
            y_min, y_max, x_min, x_max = h//2-250, h//2+250, w//2-250, w//2+250
    crop = img[y_min:y_max, x_min:x_max]
    mask_crop = mask[y_min:y_max, x_min:x_max]
    # Save to local and Drive simultaneously
    cv2.imwrite(os.path.join(LOCAL_IMG, filename), crop)
    cv2.imwrite(os.path.join(LOCAL_MASK, filename), mask_crop)
    cv2.imwrite(os.path.join(DRIVE_IMG, filename), crop)
    cv2.imwrite(os.path.join(DRIVE_MASK, filename), mask_crop)
    if idx % 100 == 0:
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); plt.title("Raw Crop"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(mask_crop, cmap='gray'); plt.title("Raw Mask"); plt.axis('off')
        plt.show()
print(f"✅ DONE! A total of {len(os.listdir(DRIVE_IMG))} cleaned images and masks are saved to Drive.")

Apply Clinical Filters & Padding

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
INPUT_DIR = "/content/new_train_data/raw_cropped"
FINAL_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
os.makedirs(FINAL_DIR, exist_ok=True)
def apply_glaucoma_filters(img):
    # 1. Extract Green Channel (Provides maximum contrast for Disc and Cup)
    if len(img.shape) == 3:
        b, g, r = cv2.split(img)
        img = g
    # 2. CLAHE (Local contrast enhancement - Critical for disc boundaries)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(img)
    # 3. Global Histogram Equalization (Brightness balance)
    return cv2.equalizeHist(clahe_img)
def pad_to_512(img):
    # Fit into 512x512 canvas with black bars instead of stretching
    h, w = img.shape[:2]
    scale = 512 / max(h, w)
    resized = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((512, 512), dtype=np.uint8)
    y_off = (512 - resized.shape[0]) // 2
    x_off = (512 - resized.shape[1]) // 2
    canvas[y_off:y_off+resized.shape[0], x_off:x_off+resized.shape[1]] = resized
    return canvas
print("🎨 Applying filters and securing 512x512 aspect ratio...")
files = os.listdir(INPUT_DIR)
for file_name in tqdm(files):
    img = cv2.imread(os.path.join(INPUT_DIR, file_name))
    if img is None: continue
    # Apply filters
    filtered_img = apply_glaucoma_filters(img)
    # Apply padding
    final_img = pad_to_512(filtered_img)
    # Save
    cv2.imwrite(os.path.join(FINAL_DIR, file_name), final_img)
print(f"\n✅ PROCESS COMPLETE! {len(files)} filtered 512x512 images saved to Drive.")

In [ ]:
import os
drive_img_target = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
drive_mask_target = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
print("🔍 ===================================================== 🔍")
print("⚡ INITIATING CLOUD STORAGE PHYSICAL FILE COUNT...\n")
if os.path.exists(drive_img_target) and os.path.exists(drive_mask_target):
    images_in_drive = len(os.listdir(drive_img_target))
    masks_in_drive = len(os.listdir(drive_mask_target))
    print(f"📸 Current Image Count in Cloud: {images_in_drive}")
    print(f"🎭 Current Mask Count in Cloud:    {masks_in_drive}")
    if images_in_drive == masks_in_drive and images_in_drive > 0:
        print("\n🚀 PERFECT! Images and masks are perfectly balanced.")
        print("   They are safely stored on disk even if Drive UI sync is delayed.")
    else:
        print("\n⚠️ ATTENTION: Mismatch detected in file counts!")
else:
    print("❌ ERROR: Specified Drive paths do not exist physically!")
print("🔍 ===================================================== 🔍")

#Load YOLO Weights

Load Pre-Trained MiT-B3 Weights

In [ ]:
import torch
# Define path to the saved model checkpoint
final_model_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v6.pth"
# Restore weights
model.load_state_dict(torch.load(final_model_path))
print("✅ 30-Epoch Base Model successfully restored. Ready for fine-tuning operations!")

Load YOLO Weights

In [ ]:
# 1. Install required library
!pip install ultralytics -q
from ultralytics import YOLO
import os
# 2. Verify Drive connection
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
# 3. Load the YOLO Model
model_path = '/content/drive/MyDrive/yeni_glokom_proje/yolo_save/yolo_disk_dedektor_yeniv3.pt'
yolo_model = YOLO(model_path)
print("✅ YOLO Model successfully loaded from Drive!")

# Data Augmentations (Base)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
# Base Transformation pipeline to prevent overfitting during training
train_transform = A.Compose([
    # 1. Geometric (Neutralize angle and position variance)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.5),
    # 2. Lighting and Style (Prevent washouts or extreme darkness)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    # 3. Sharpness and Noise (Build immunity against varied clinical filters)
    A.Sharpen(alpha=(0.2, 0.4), p=0.3),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
    # 4. Normalization (Optimal ImageNet values for MiT-B3)
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

Data Augmentations (Advanced SOTA Features)

In [ ]:

import albumentations as A
from albumentations.pytorch import ToTensorV2
train_transform = A.Compose([
    # 1. Geometric adjustments
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.5),
    # 2. SOTA FEATURES: Biological retinal stretching and deformations
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    A.GridDistortion(p=0.5),
    # 3. Lighting and Style
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    # 4. Sharpness and Noise
    A.Sharpen(alpha=(0.2, 0.4), p=0.3),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
    # 5. Normalization (Must remain at the very end)
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

Data Augmentations (Fine-Tuning)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
# OPTIMIZED TRANSFORM FOR FINE-TUNING
train_transform = A.Compose([
    # 1. Anatomical symmetry
    A.HorizontalFlip(p=0.5),
    # 2. Minor brightness/contrast adjustments
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    # 3. Sharpness
    A.Sharpen(alpha=(0.1, 0.2), p=0.2),
    # 4. NORMALIZATION
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False) # <--- Parameter added to resolve dimensionality constraints
print("✅ Fine-tuning transformation pipeline successfully configured.")

Data Augmentations (Grayscale Optimized)"

In [ ]:

import albumentations as A
from albumentations.pytorch import ToTensorV2
# Grayscale (1-Channel) SOTA specific, mask-safe transformation pipeline
train_transform = A.Compose([
    # 1. Geometric adjustments
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.5),
    # 2. SOTA FEATURES: Biological retinal stretching and deformations
    # Interpolation locked to avoid corrupting mask values (0,1,2)
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    A.GridDistortion(p=0.5),
    # 3. Lighting and Style
    # HueSaturationValue strictly removed for Grayscale stability.
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    # 4. Sharpness and Noise
    A.Sharpen(alpha=(0.2, 0.4), p=0.3),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
    # 5. OUTPUT (1-Channel normalization for MiT-B3 architecture)
    A.Normalize(mean=(0.485,), std=(0.229,)),
    ToTensorV2()
])
print("✅ Grayscale SOTA transformations successfully configured!")

YOLO Specific Augmentations

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
# YOLO Fine-Tuning constraints
train_transform_yolo = A.Compose([
    # 2. LIGHTING AND COLOR (Simulate sensor variance across devices)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    # 3. SHARPNESS AND NOISE (Simulate camera/lens degradation)
    A.Sharpen(alpha=(0.2, 0.4), p=0.3),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
    # 4. OUTPUT (Normalization removed, ToTensorV2 retained)
    # YOLO handles internal 0-1 normalization natively.
    ToTensorV2()
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Dataset Loader (No Augmentation)

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
# 1. RESTORE BASE MODEL
final_model_path = '/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_final_v1.pth'
model.load_state_dict(torch.load(final_model_path))
print("✅ 30-Epoch Base Model Restored!")
# 2. DATASET CLASS CONFIGURATION (With Built-in Normalization)
class GlaucomaDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted(os.listdir(img_dir))
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        mask_path = os.path.join(self.mask_dir, self.img_list[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        # MiT-B3 Strict Normalization
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = (img - mean) / std
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        img = torch.from_numpy(img).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).long()
        return img, mask
# 3. DIRECTORY DEFINITIONS
ASIL_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/train_data/images"
ASIL_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/train_data/masks"
# Initialize Dataloaders
train_ds = GlaucomaDataset(ASIL_IMG_DIR, ASIL_MASK_DIR)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=8, pin_memory=True)
print(f"✅ DataLoader ready! {len(train_ds)} clean samples loaded.")

Dataset Loader (With Augmentation)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
class GlaucomaDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted(os.listdir(img_dir))
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        mask_path = os.path.join(self.mask_dir, self.img_list[idx])
        # Read Image (BGR -> RGB)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Read Mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            # Manual tensor conversion if transform is None
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1)
            mask = torch.from_numpy(mask)
        return img, mask.long()
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/train_data/masks"
train_ds = GlaucomaDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
# A100 Turbo Configuration
train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=8,     # 8 workers optimal for A100 throughput
    pin_memory=True    # Accelerates GPU memory transfer
)
print(f"✅ Transform-supported DataLoader ready! Total samples: {len(train_ds)}")

Test Dataset Loader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
# 2. EVALUATION DATASET CLASS
class GlaucomaDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        # List only image files
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        # ⚠️ CRITICAL: Replace '_img' with '_mask' to accurately locate corresponding mask
        mask_name = img_name.replace("_img", "_mask")
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        # MiT-B3 Normalization
        mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
        img = (img - mean) / std
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        # 🛡️ SAFETY CHECK: Throw descriptive error instead of silent crash
        if mask is None:
            raise FileNotFoundError(f"❌ Mask not found! Path: {mask_path}")
        img = torch.from_numpy(img).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).long()
        return img, mask
# 3. PATH DEFINITIONS
ASIL_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/images"
ASIL_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/masks"
# Dataset and Loader (num_workers=0 to trace potential errors clearly)
train_ds = GlaucomaDataset(ASIL_IMG_DIR, ASIL_MASK_DIR)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
print(f"✅ Test Data Pipeline Ready! {len(train_ds)} evaluation samples loaded.")


YOLO Dataset Loader (No Augmentation)

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
class GlaucomaFineTuneDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        # Fetch intersection of files to prevent mismatches
        img_names = set(os.listdir(img_dir))
        mask_names = set(os.listdir(mask_dir))
        self.img_list = sorted(list(img_names.intersection(mask_names)))
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        mask_path = os.path.join(self.mask_dir, self.img_list[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1)
            mask = torch.from_numpy(mask)
        return img, mask.long()
# --- DEFINE NEW DRIVE DATASETS ---
NEW_IMG_DIR = "/content/drive/MyDrive/glokom_proje/new_dataset/images"
NEW_MASK_DIR = "/content/drive/MyDrive/glokom_proje/new_dataset/masks"
train_ds = GlaucomaFineTuneDataset(NEW_IMG_DIR, NEW_MASK_DIR, transform=None)
# Turbo Mode DataLoader
train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2
)
print(f"🚀 Turbo DataLoader ready! {len(train_ds)} samples processing on A100.")

YOLO Dataset Loader (With Augmentation)

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np

class GlaucomaFineTuneDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        img_names = set(os.listdir(img_dir))
        mask_names = set(os.listdir(mask_dir))
        self.img_list = sorted(list(img_names.intersection(mask_names)))
        self.transform = transform

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        mask_path = os.path.join(self.mask_dir, self.img_list[idx])

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1)
            mask = torch.from_numpy(mask)

        return img, mask.long()

NEW_IMG_DIR = "/content/drive/MyDrive/glokom_proje/new_dataset/images"
NEW_MASK_DIR = "/content/drive/MyDrive/glokom_proje/new_dataset/masks"
# Similar to above, but transform argument is active
train_ds = GlaucomaFineTuneDataset(NEW_IMG_DIR, NEW_MASK_DIR, transform=train_transform)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
print(f"✅ Drive-focused DataLoader ready! Total of {len(train_ds)} cropped samples to process.")

Turbo DataLoader (A100 Optimized)

In [ ]:

import os
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
# 1. DATASET CLASS
class GlaucomaFineTuneDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        img_names = set(os.listdir(img_dir))
        mask_names = set(os.listdir(mask_dir))
        self.img_list = sorted(list(img_names.intersection(mask_names)))
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        mask_path = os.path.join(self.mask_dir, self.img_list[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1)
            mask = torch.from_numpy(mask)
        return img, mask.long()
# 2. TRANSFORM DEFINITION
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
# 3. DIRECTORIES
NEW_IMG_DIR = "/content/drive/MyDrive/glokom_proje/new_dataset/images"
NEW_MASK_DIR = "/content/drive/MyDrive/glokom_proje/new_dataset/masks"
# 4. DATALOADER INITIALIZATION
train_ds = GlaucomaFineTuneDataset(NEW_IMG_DIR, NEW_MASK_DIR, transform=train_transform)
train_loader = DataLoader(
    train_ds,
    batch_size=32,       # Optimal batch size for A100
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2
)
print(f"🚀 HYBRID SYSTEM ACTIVATED!")
print(f"✅ Speed: A100 Turbo (Batch: 32, Workers: 8)")
print(f"✅ Intelligence: Transforms active to prevent memorization.")
print(f"📦 Total {len(train_ds)} clinical samples will be processed.")

Grayscale Dataset Loader (REFUGE2 & DRISHTI)

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
# ==============================================================================
# 🩻 2. REFUGE2 & DRISHTI COMPATIBLE GRAYSCALE DATASET
# ==============================================================================
class GlaucomaGrayscaleDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        # Read directly in Grayscale SOTA format
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            # Firewall: Return dummy tensors to prevent system crash on corrupted files
            return torch.zeros((1, 512, 512), dtype=torch.float32), torch.zeros((512, 512), dtype=torch.long)
        # Albumentations expects 3D matrices [H, W, 1] for single-channel images
        img = np.expand_dims(img, axis=-1)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask)
        return img, mask.long()
# ==============================================================================
# 🚀 3. TURBO DATALOADER CONFIGURATION
# ==============================================================================
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
train_ds = GlaucomaGrayscaleDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
# Maximize A100 GPU VRAM potential
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2,
    drop_last=True
)
print("🔍 ===================================================== 🔍")
print("🚀 SOTA HYBRID DATALOADER SYSTEM INITIALIZED!")
print("-" * 56)
print(f"✅ Speed Mode: A100 Turbo Active (Batch: 32, Workers: 8)")
print(f"✅ Input Channel: 1-Channel Mode (Grayscale)")
print(f"✅ Intelligence: Retinal Augmentations Integrated.")
print(f"📦 Total Clinical Training Samples: {len(train_ds)}")
print("🔍 ===================================================== 🔍")

# Extract Temporary ZIP from Drive

In [ ]:
import os
# 1. Drive ZIP Path
zip_path = "/content/drive/MyDrive/yeni_glokom_proje/train_data/process_data.zip"
# 2. Prepare temporary workspace
!rm -rf /content/data
os.makedirs("/content/data", exist_ok=True)
# 3. Unpack ZIP to fast Colab memory
!unzip -q "{zip_path}" -d /content/data
print("✅ ZIP successfully unpacked from Drive!")
print("Extracted directories:", os.listdir("/content/data"))

YOLO No-Stretch Padding

In [ ]:
import cv2
import os
import torch
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
# --- 🎯 PATH DEFINITIONS ---
DRIVE_BASE = "/content/drive/MyDrive/yeni_glokom_proje"
YOLO_WEIGHTS = f"{DRIVE_BASE}/yolo_save/yolo_disk_dedektor_yeni.pt"
# Permanent repository for MiT-B3 training
OUTPUT_IMG_DIR = f"{DRIVE_BASE}/train_data/images"
OUTPUT_MASK_DIR = f"{DRIVE_BASE}/train_data/masks"
ORIGINAL_IMAGES_DIR = "/content/data/images"
ORIGINAL_MASKS_DIR = "/content/data/masks"
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
# Load Model
yolo_model = YOLO(YOLO_WEIGHTS)
# --- 🛡️ CLINICAL PADDING (Black Bars) ---
def apply_padding(image, target_size=512, is_mask=False):
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    # Use NEAREST for masks to preserve class IDs, LINEAR for images
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    resized = cv2.resize(image, (new_w, new_h), interpolation=interp)
    canvas_shape = (target_size, target_size, 3) if not is_mask else (target_size, target_size)
    canvas = np.zeros(canvas_shape, dtype=np.uint8)
    # Center alignment
    x_off = (target_size - new_w) // 2
    y_off = (target_size - new_h) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return canvas
print(f"🚀 Operation Initiated: Processing raw data to {OUTPUT_IMG_DIR}...")
image_files = [f for f in os.listdir(ORIGINAL_IMAGES_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))]
for file_name in tqdm(image_files):
    img_path = os.path.join(ORIGINAL_IMAGES_DIR, file_name)
    mask_path = os.path.join(ORIGINAL_MASKS_DIR, file_name)
    if not os.path.exists(mask_path): continue
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: continue
    h, w, _ = img.shape
    # 1. YOLO Optic Disc Detection
    results = yolo_model.predict(img, conf=0.5, verbose=False)
    if len(results[0].boxes) > 0:
        box = results[0].boxes.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, box)
        # 15% Safety Margin (Padding)
        pad_w, pad_h = int((x2 - x1) * 0.15), int((y2 - y1) * 0.15)
        x1, y1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
        x2, y2 = min(w, x2 + pad_w), min(h, y2 + pad_h)
        # 2. Raw Cropping
        cropped_img = img[y1:y2, x1:x2]
        cropped_mask = mask[y1:y2, x1:x2]
        # 3. Square Canvas Alignment (512x512 with black bars)
        final_img = apply_padding(cropped_img, target_size=512, is_mask=False)
        final_mask = apply_padding(cropped_mask, target_size=512, is_mask=True)
        # 4. Save to Drive Storage
        cv2.imwrite(os.path.join(OUTPUT_IMG_DIR, file_name), final_img)
        cv2.imwrite(os.path.join(OUTPUT_MASK_DIR, file_name), final_mask)
print(f"\n✅ SUCCESS! New data cropped, padded, and added to the Drive repository.")

Extract Test Data ZIP

In [ ]:
import os
# 1. Test data ZIP path
zip_path = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_data2.zip"
# 2. Clean workspace
!rm -rf /content/data
os.makedirs("/content/data", exist_ok=True)
# 3. Extract ZIP
!unzip -q "{zip_path}" -d /content/data
print("✅ ZIP successfully unpacked!")
print("Extracted files (first 5):", os.listdir("/content/data")[:5])

Generate Test Data Pipeline

In [ ]:
import cv2
import os
import torch
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
# --- 🎯 STEP 1: DIRECTORIES ---
TEST_ZIP_PATH = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_data2.zip"
DRIVE_BASE = "/content/drive/MyDrive/yeni_glokom_proje"
YOLO_WEIGHTS = f"{DRIVE_BASE}/yolo_save/yolo_disk_dedektor_yeni.pt"
TEMP_DATA_DIR = "/content/data_test_temp"
!rm -rf {TEMP_DATA_DIR}
os.makedirs(TEMP_DATA_DIR, exist_ok=True)
print(f"📦 Unpacking ZIP: {TEST_ZIP_PATH}...")
!unzip -q "{TEST_ZIP_PATH}" -d {TEMP_DATA_DIR}
# Folder Wizard
files_in_temp = os.listdir(TEMP_DATA_DIR)
if len(files_in_temp) == 1 and os.path.isdir(os.path.join(TEMP_DATA_DIR, files_in_temp[0])):
    ORIGINAL_TEST_DIR = os.path.join(TEMP_DATA_DIR, files_in_temp[0])
else:
    ORIGINAL_TEST_DIR = TEMP_DATA_DIR
OUTPUT_TEST_IMAGE_DIR = f"{DRIVE_BASE}/test_data/test_image"
os.makedirs(OUTPUT_TEST_IMAGE_DIR, exist_ok=True)
# --- 🛡️ STEP 2: MODEL SETUP ---
yolo_model = YOLO(YOLO_WEIGHTS)
def apply_padding_only_img(image, target_size=512):
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    x_off = (target_size - new_w) // 2
    y_off = (target_size - new_h) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return canvas
# --- 🚀 STEP 3: EXECUTION ---
print(f"🔥 Processing initiated! Source: {ORIGINAL_TEST_DIR}")
image_files = [f for f in os.listdir(ORIGINAL_TEST_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
if len(image_files) == 0:
    print("❌ ERROR: No images found in directory! Verify path.")
else:
    for file_name in tqdm(image_files):
        img_path = os.path.join(ORIGINAL_TEST_DIR, file_name)
        img = cv2.imread(img_path)
        if img is None: continue
        h, w, _ = img.shape
        results = yolo_model.predict(img, conf=0.4, verbose=False)
        if len(results[0].boxes) > 0:
            box = results[0].boxes.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            pad_w, pad_h = int((x2 - x1) * 0.15), int((y2 - y1) * 0.15)
            x1, y1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
            x2, y2 = min(w, x2 + pad_w), min(h, y2 + pad_h)
            cropped_img = img[y1:y2, x1:x2]
            final_img = apply_padding_only_img(cropped_img, target_size=512)
            cv2.imwrite(os.path.join(OUTPUT_TEST_IMAGE_DIR, file_name), final_img)
    print(f"\n✅ SUCCESS! {len(image_files)} images processed and saved.")

Extract Masked Test Data ZIP

In [ ]:
import os
zip_path = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_data3.zip"
!rm -rf /content/test_data_raw
os.makedirs("/content/test_data_raw", exist_ok=True)
print(f"📦 Extracting test data: {zip_path}...")
!unzip -q "{zip_path}" -d /content/test_data_raw
print("✅ ZIP successfully unpacked!")
print("Contents:", os.listdir("/content/test_data_raw"))

Generate Masked Test Data

In [ ]:
!pip install ultralytics -q
import cv2
import os
import torch
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
DRIVE_BASE = "/content/drive/MyDrive/yeni_glokom_proje"
YOLO_WEIGHTS = f"{DRIVE_BASE}/yolo_save/yolo_disk_dedektor_yeni.pt"
OUTPUT_IMG_DIR = f"{DRIVE_BASE}/test_data/test_withmask/images"
OUTPUT_MASK_DIR = f"{DRIVE_BASE}/test_data/test_withmask/masks"
ORIGINAL_IMAGES_DIR = "/content/test_data_raw/images"
ORIGINAL_MASKS_DIR = "/content/test_data_raw/masks"
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
yolo_model = YOLO(YOLO_WEIGHTS)
def apply_padding(image, target_size=512, is_mask=False):
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    resized = cv2.resize(image, (new_w, new_h), interpolation=interp)
    canvas_shape = (target_size, target_size, 3) if not is_mask else (target_size, target_size)
    canvas = np.zeros(canvas_shape, dtype=np.uint8)
    x_off = (target_size - new_w) // 2
    y_off = (target_size - new_h) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return canvas
print(f"🚀 Processing test data into -> {OUTPUT_IMG_DIR}")
image_files = [f for f in os.listdir(ORIGINAL_IMAGES_DIR) if "_img" in f and f.lower().endswith(('.png', '.jpg', '.jpeg'))]
for file_name in tqdm(image_files):
    mask_name = file_name.replace("_img", "_mask") # Naming convention handler
    img_path = os.path.join(ORIGINAL_IMAGES_DIR, file_name)
    mask_path = os.path.join(ORIGINAL_MASKS_DIR, mask_name)
    if not os.path.exists(mask_path): continue
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: continue
    h, w, _ = img.shape
    results = yolo_model.predict(img, conf=0.5, verbose=False)
    if len(results[0].boxes) > 0:
        box = results[0].boxes.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, box)
        pad_w, pad_h = int((x2 - x1) * 0.15), int((y2 - y1) * 0.15)
        x1, y1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
        x2, y2 = min(w, x2 + pad_w), min(h, y2 + pad_h)
        cropped_img = img[y1:y2, x1:x2]
        cropped_mask = mask[y1:y2, x1:x2]
        final_img = apply_padding(cropped_img, target_size=512, is_mask=False)
        final_mask = apply_padding(cropped_mask, target_size=512, is_mask=True)
        cv2.imwrite(os.path.join(OUTPUT_IMG_DIR, file_name), final_img)
        cv2.imwrite(os.path.join(OUTPUT_MASK_DIR, mask_name), final_mask)
print(f"\n✅ SUCCESS! Masked test dataset compiled.")

Extract New Test Data ZIP and Save to Mask Dir

In [ ]:
import os, zipfile, shutil, cv2, numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
DRIVE_IMG = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/images"
DRIVE_MASK = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/masks"
TEMP_DIR = "/content/test_temp_raw"
ZIP_PATH = "/content/drive/MyDrive/yeni_glokom_proje/test_data/TEST.zip"
for path in [DRIVE_IMG, DRIVE_MASK]:
    if os.path.exists(path): shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
if os.path.exists(TEMP_DIR): shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)
print("📦 Unpacking ZIP...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z: z.extractall(TEMP_DIR)
def find_folder(base, name):
    for root, dirs, files in os.walk(base):
        if name in dirs: return os.path.join(root, name)
    return None
IMAGES_DIR = find_folder(TEMP_DIR, "images")
MASKS_DIR = find_folder(TEMP_DIR, "masks")
img_files = {os.path.splitext(f)[0]: f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))}
mask_files = {os.path.splitext(f)[0]: f for f in os.listdir(MASKS_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))}
common_keys = set(img_files.keys()) & set(mask_files.keys())
print(f"🚀 Total of {len(common_keys)} matching images found!")
yolo_model = YOLO('/content/drive/MyDrive/yeni_glokom_proje/yolo_save/yolo_disk_dedektor_yeniv2.pt')
for idx, key in enumerate(tqdm(common_keys)):
    img_name = img_files[key]
    mask_name = mask_files[key]
    img = cv2.imread(os.path.join(IMAGES_DIR, img_name))
    mask = cv2.imread(os.path.join(MASKS_DIR, mask_name), cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: continue
    h, w = img.shape[:2]
    is_refuge = "refuge" in img_name.lower() or max(h, w) > 1200
    if is_refuge:
        pts = np.argwhere(mask > 0)
        if len(pts) > 0:
            cy, cx = int(np.mean(pts[:, 0])), int(np.mean(pts[:, 1]))
            y_min, y_max = max(0, cy - 350), min(h, cy + 350)
            x_min, x_max = max(0, cx - 350), min(w, cx + 350)
        else:
            y_min, y_max, x_min, x_max = h//2-350, h//2+350, w//2-350, w//2+350
    else:
        results = yolo_model.predict(img, conf=0.4, verbose=False)
        if len(results[0].boxes) > 0:
            x1, y1, x2, y2 = map(int, results[0].boxes.xyxy[0])
            cy, cx = (y1 + y2) // 2, (x1 + x2) // 2
            # --- ZOOM-OUT: 650 offset for expanded field of view ---
            offset = 650
            y_min, y_max = max(0, cy - offset), min(h, cy + offset)
            x_min, x_max = max(0, cx - offset), min(w, cx + offset)
        else:
            y_min, y_max, x_min, x_max = h//2-350, h//2+350, w//2-350, w//2+350
    crop = img[y_min:y_max, x_min:x_max]
    mask_crop = mask[y_min:y_max, x_min:x_max]
    cv2.imwrite(os.path.join(DRIVE_IMG, f"{key}.png"), crop)
    cv2.imwrite(os.path.join(DRIVE_MASK, f"{key}.png"), mask_crop)
    if idx % 100 == 0:
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); plt.title("Wide FoV (Zoom-Out)"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(mask_crop, cmap='gray'); plt.title("Ground Truth Mask"); plt.axis('off')
        plt.show()
shutil.rmtree(TEMP_DIR)
print(f"✅ DONE! {len(os.listdir(DRIVE_IMG))} processed files saved to Drive.")

Apply Test Set Filters

In [ ]:

import os, cv2, numpy as np
from tqdm import tqdm
INPUT_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/images"
MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/masks"
def apply_glaucoma_filters(img):
    if len(img.shape) == 3:
        img = img[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(img)
    return cv2.equalizeHist(clahe_img)
def pad_to_512(img, is_mask=False):
    h, w = img.shape[:2]
    scale = 512 / max(h, w)
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=interp)
    canvas = np.zeros((512, 512), dtype=np.uint8)
    y_off = (512 - new_h) // 2
    x_off = (512 - new_w) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return canvas
print("🛠️ Process initiated: Filtering and standardizing to 512x512...")
files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
for filename in tqdm(files):
    img_path = os.path.join(INPUT_DIR, filename)
    mask_path = os.path.join(MASK_DIR, filename)
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        continue
    filtered_img = apply_glaucoma_filters(img)
    final_img = pad_to_512(filtered_img, is_mask=False)
    final_mask = pad_to_512(mask, is_mask=True)
    cv2.imwrite(img_path, final_img)
    cv2.imwrite(mask_path, final_mask)
print(f"\n✅ PROCESS COMPLETE! {len(files)} test images updated.")


Verification Query

In [ ]:
import os
target_dir = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask"
img_path = os.path.join(target_dir, "images")
mask_path = os.path.join(target_dir, "masks")
if os.path.exists(img_path) and os.path.exists(mask_path):
    img_count = len([f for f in os.listdir(img_path) if not f.startswith('.')])
    mask_count = len([f for f in os.listdir(mask_path) if not f.startswith('.')])
    print(f"📊 Audit Results:")
    print(f"📁 Files in 'images': {img_count}")
    print(f"📁 Files in 'masks': {mask_count}")
    if img_count == mask_count:
        print("✅ Excellent! Symmetrical structure confirmed.")
    else:
        print("⚠️ WARNING! Asymmetry detected. Data missing.")
else:
    print("❌ Target directories not found! Validate ZIP extraction pipeline.")


# Training

Initial Training Phase (MiT-B3)

In [ ]:
from tqdm import tqdm
num_epochs = 30
model.train()
print("🚀 Training sequence initiated... A100 engine active.")
for epoch in range(num_epochs):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    for images, masks in loop:
        images = images.cuda()
        masks = masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        save_path = f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_v3_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Checkpoint secured: {save_path}")
final_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v5.pth"
torch.save(model.state_dict(), final_path)
print(f"\n🎉 OPERATION COMPLETE! Final model located at: {final_path}")
print("\n🎉 Congratulations! Model training successfully concluded.")

Fixed Configuration for New Dataset

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
# ==============================================================================
# 🛡️ STEP 1: DIMENSIONAL-GUARANTEE DATASET CLASS
# ==============================================================================
class GlaucomaGrayscaleDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            return torch.zeros((1, 512, 512), dtype=torch.float32), torch.zeros((512, 512), dtype=torch.long)
        # 🚨 DIMENSION SYNC PROTOCOL (Prevents runtime crash)
        if img.shape != mask.shape:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask[mask > 2] = 0 # Out-of-bounds artifact cleaning
        img = np.expand_dims(img, axis=-1)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask)
        return img, mask.long()
# ==============================================================================
# 🚀 STEP 2: TURBO DATALOADER
# ==============================================================================
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
train_ds = GlaucomaGrayscaleDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
# ==============================================================================
# 🔥 STEP 3: TRAINING LOOP
# ==============================================================================
torch.cuda.empty_cache()
num_epochs = 30
model.train()
print("🚀 TRAINING INITIATED... A100 FULL THROTTLE!")
for epoch in range(num_epochs):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    for images, masks in loop:
        images = images.cuda()
        masks = masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        save_path = f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_v8_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Checkpoint secured: {save_path}")
final_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v8.pth"
torch.save(model.state_dict(), final_path)
print(f"\n🎉 OPERATION COMPLETE! Final model located at: {final_path}")

Fallback / Alternative Implementation (RGB Merge trick)

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
# ==============================================================================
# 🛡️ STEP 1: DIMENSIONAL-GUARANTEE 3-CHANNEL DATASET CLASS
# ==============================================================================
class GlaucomaGrayscaleDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        # Fallback to 3 channels to prevent crash
        if img is None or mask is None:
            return torch.zeros((3, 512, 512), dtype=torch.float32), torch.zeros((512, 512), dtype=torch.long)
        # 🚨 DIMENSION SYNC PROTOCOL (INTER_NEAREST is strictly required!)
        if img.shape != mask.shape:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask[mask > 2] = 0
        # 🔥 STRUCTURAL ADAPTATION: Replicating 1-channel Grayscale into pseudo-RGB layers!
        img = cv2.merge([img, img, img])
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask)
        return img, mask.long()
# ==============================================================================
# 🚀 STEP 2: TURBO DATALOADER
# ==============================================================================
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
train_ds = GlaucomaGrayscaleDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
# ==============================================================================
# 🔥 STEP 3: TRAINING LOOP
# ==============================================================================
torch.cuda.empty_cache()
num_epochs = 30
model.train()
print("🚀 TRAINING INITIATED... A100 FULL THROTTLE!")
for epoch in range(num_epochs):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    for images, masks in loop:
        images = images.cuda()
        masks = masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        save_path = f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_v7_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Checkpoint secured: {save_path}")
final_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v7.pth"
torch.save(model.state_dict(), final_path)
print(f"\n🎉 OPERATION COMPLETE! Final model located at: {final_path}")


MiT-B3 Fine-Tune Phase 1

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
# 1. ERROR-FREE TRANSFORM CONFIGURATION (is_check_shapes=False implemented)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.Sharpen(alpha=(0.1, 0.2), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)
# 2. DIMENSIONAL-GUARANTEE DATASET
class GlaucomaGrayscaleDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        img = cv2.imread(os.path.join(self.img_dir, img_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, img_name), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            return torch.zeros((1, 512, 512)), torch.zeros((512, 512), dtype=torch.long)
        # STRICT DIMENSIONAL FIXATION (Resized prior to transformation logic)
        img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
        mask[mask > 2] = 0
        img = np.expand_dims(img, axis=-1)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        else:
            img = torch.from_numpy(img.astype(np.float32)/255.0).permute(2,0,1)
            mask = torch.from_numpy(mask)
        return img, mask.long()
# 3. DATALOADER (num_workers=0 to trace potential process blocks)
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
train_ds = GlaucomaGrayscaleDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
# 4. TRAINING SEQUENCE
model.train()
print("🚀 Training initiated...")
for epoch in range(10):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/10]", leave=True)
    for images, masks in loop:
        images, masks = images.cuda(), masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_Finetune_v7_epoch{epoch+1}.pth")
torch.save(model.state_dict(), "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v7.pth")
print("\n🎉 OPERATION COMPLETE!")

MiT-B3 Fine-Tune Phase 2 (VRAM Optimized)

In [ ]:
import os
import gc
import torch
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast
# 1. SYSTEM RESET
torch.cuda.empty_cache()
gc.collect()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# 2. DATASET
class GlaucomaGrayscaleDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        img = cv2.imread(os.path.join(self.img_dir, img_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, img_name), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            return torch.zeros((1, 512, 512)), torch.zeros((512, 512), dtype=torch.long)
        mask[mask > 2] = 0
        img = np.expand_dims(img, axis=-1)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        else:
            img = torch.from_numpy(img.astype(np.float32)/255.0).permute(2,0,1)
            mask = torch.from_numpy(mask)
        return img, mask.long()
# 3. DATALOADER
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
train_ds = GlaucomaGrayscaleDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
# 4. FINAL TRAINING SEQUENCE (VRAM Protected)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
scaler = GradScaler()
num_epochs = 10
print("🚀 PRISTINE TRAINING SEQUENCE INITIATED...")
for epoch in range(num_epochs):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]")
    for images, masks in loop:
        images, masks = images.cuda(), masks.cuda()
        optimizer.zero_grad(set_to_none=True)
        with autocast(): # Half-precision logic for massive memory savings
            outputs = model(images)
            loss = calculate_loss(outputs, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.detach().item()
        loop.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        save_path = f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_v7_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Checkpoint secured: {save_path}")
print("\n🎉 OPERATION COMPLETE!")
# FINAL SAVE
final_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v5_finetune.pth"
torch.save(model.state_dict(), final_path)
print(f"\n🎉 OPERATION COMPLETE! Final model located at: {final_path}")

MiT-B3 Deep Polishing (Focal+Dice Custom)

In [ ]:
from tqdm import tqdm
import os
# --- POLISHING CONFIGURATION ---
num_epochs_cila = 10
model.train()
# Low learning rate for precise fine-tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
print("✨ Polishing Sequence Initiated: Hybrid Focal+Dice Loss is active...")
for epoch in range(num_epochs_cila):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Polish Epoch [{epoch+1}/{num_epochs_cila}]", leave=True)
    for images, masks in loop:
        images, masks = images.cuda(), masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        # Utilizing the custom HybridLoss formulation
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"✨ Polish Epoch [{epoch+1}/{num_epochs_cila}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        cila_save_path = f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_CILALI_v3_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), cila_save_path)
        print(f"💎 Polish Checkpoint Acquired: {cila_save_path}")
# FINAL CHECKPOINT
final_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_arasonrası_finetune.pth"
torch.save(model.state_dict(), final_path)
print(f"\n🎉 OPERATION COMPLETE! Final polished model at: {final_path}")

YOLO Label Generation and Formatting (Pre-Training Phase)

In [ ]:
import os
import cv2
import shutil
import numpy as np
from tqdm import tqdm
# 1. PATH DEFINITIONS
ORIGINAL_IMAGES = "/content/data/images"
ORIGINAL_MASKS = "/content/data/masks"
YOLO_ENV = "/content/yolo_env"
# Reset directories
!rm -rf {YOLO_ENV}
os.makedirs(f"{YOLO_ENV}/images", exist_ok=True)
os.makedirs(f"{YOLO_ENV}/labels", exist_ok=True)
# 2. CLINICAL FILTERING
def apply_glaucoma_filters(img):
    if len(img.shape) == 3:
        b, g, r = cv2.split(img)
        img = g
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(img)
    return cv2.equalizeHist(clahe_img)
print(f"🔄 Generating YOLO labels and applying clinical filters to images...")
count = 0
for file_name in tqdm(os.listdir(ORIGINAL_MASKS)):
    if not file_name.endswith(('.png', '.jpg', '.jpeg', '.bmp')): continue
    mask_path = os.path.join(ORIGINAL_MASKS, file_name)
    img_path = os.path.join(ORIGINAL_IMAGES, file_name)
    # Name matching logic
    if not os.path.exists(img_path):
        basename_no_ext = os.path.splitext(file_name)[0]
        bulundu = False
        for ext in ['.png', '.jpg', '.jpeg', '.bmp']:
            test_path = os.path.join(ORIGINAL_IMAGES, basename_no_ext + ext)
            if os.path.exists(test_path):
                img_path = test_path
                file_name = basename_no_ext + ext
                bulundu = True
                break
        if not bulundu: continue
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.imread(img_path)
    if mask is None or img is None: continue
    pts = np.argwhere(mask > 0)
    if len(pts) > 0:
        ymin, xmin = np.min(pts, axis=0)
        ymax, xmax = np.max(pts, axis=0)
        box_w, box_h = xmax - xmin, ymax - ymin
        img_h, img_w = mask.shape
        # 15% Safe Padding Limit
        pad_w, pad_h = int(box_w * 0.15), int(box_h * 0.15)
        x1, y1 = max(0, xmin - pad_w), max(0, ymin - pad_h)
        x2, y2 = min(img_w, xmax + pad_w), min(img_h, ymax + pad_h)
        # 🎯 Normalize coordinates for YOLO format
        x_center = ((x1 + x2) / 2.0) / img_w
        y_center = ((y1 + y2) / 2.0) / img_h
        width = (x2 - x1) / img_w
        height = (y2 - y1) / img_h
        # 1. SAVE LABEL
        txt_name = os.path.splitext(file_name)[0] + ".txt"
        with open(f"{YOLO_ENV}/labels/{txt_name}", "w") as f:
            f.write(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
        # 2. FILTER AND SAVE IMAGE
        filtered_img = apply_glaucoma_filters(img)
        cv2.imwrite(os.path.join(f"{YOLO_ENV}/images", file_name), filtered_img)
        count += 1
# YAML Configuration Logic
yaml_text = f"path: {YOLO_ENV}\ntrain: images\nval: images\nnames:\n  0: Optic_Disk"
with open("/content/glokom_yolo.yaml", "w") as f: f.write(yaml_text)
print(f"\n✅ PROCESS COMPLETE! {count} filtered images securely mapped to YOLO_ENV.")

YOLOv8 Core Training Initialization

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
# Load Base Nano Architecture
model_yolo = YOLO('yolov8n.pt')
# Initiate Base Training Protocol
model_yolo.train(
    data='/content/glokom_yolo.yaml',
    epochs=30,
    imgsz=640,
    name='optik_disk_bulucu'
)

YOLO Fine-Tuning Execution

In [ ]:
from tqdm import tqdm
num_epochs = 10
model.train()
model.cuda()
print("🚀 Fine-Tuning operation initialized... MiT-B3 is now focusing on close-up details.")
for epoch in range(num_epochs):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    for images, masks in loop:
        images = images.cuda()
        masks = masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(batch_loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"📈 Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        save_path = f"/content/drive/MyDrive/glokom_proje/glaucoma_finetuned_v2_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Checkpoint safely exported to Drive: {save_path}")
print("\n🎉 Congratulations! The close-up specialization parameters are optimally integrated.")

# Model Weight Saving Operations

In [ ]:
final_model_path = '/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_final_v2.pth'
# Save state dictionary
torch.save(model.state_dict(), final_model_path)
print("✅ MODEL SECURELY SAVED TO DRIVE. MISSION ACCOMPLISHED!")

YOLO Checkpoint Archiving

In [ ]:

import shutil
import os
# YOLOv8 default generation pathway
source = '/content/runs/detect/optik_disk_bulucu/weights/best.pt'
destination_folder = '/content/drive/MyDrive/glokom_proje'
destination_file = os.path.join(destination_folder, 'yolo_disk_dedektor_yeniv3.pt')
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
if os.path.exists(source):
    shutil.copy(source, destination_file)
    print(f"✅ Model successfully secured and archived at: {destination_file}")
else:
    print("❌ ERROR: 'best.pt' file not found. Ensure training protocol concluded completely.")

Save Final YOLO Weights (Secondary Folder)

In [ ]:

import shutil
import os
source = '/content/runs/detect/optik_disk_bulucu/weights/best.pt'
destination_folder = '/content/drive/MyDrive/yeni_glokom_proje/yolo_save'
destination_file = os.path.join(destination_folder, 'yolo_disk_dedektor_yeniv2gem.pt')
os.makedirs(destination_folder, exist_ok=True)
if os.path.exists(source):
    shutil.copy(source, destination_file)
    print(f"✅ New YOLO iteration successfully archived to Drive!\nLocation: {destination_file}")
else:
    print("❌ ERROR: 'best.pt' not found. Training cycle may be incomplete.")

# Visualization & Performance Evaluation

In [ ]:
import os, cv2, torch, numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
# --- 1. UTILITY FUNCTIONS ---
def calculate_dice(pred, mask, label):
    pred_bin = (pred == label).astype(np.float32)
    mask_bin = (mask == label).astype(np.float32)
    intersection = np.sum(pred_bin * mask_bin)
    return (2.0 * intersection) / (np.sum(pred_bin) + np.sum(mask_bin) + 1e-7)
def get_cat(score):
    s = score * 100
    if s >= 90: return "100-90"
    elif s >= 85: return "90-85"
    elif s >= 80: return "85-80"
    elif s >= 75: return "80-75"
    elif s >= 70: return "75-70"
    elif s >= 60: return "70-60"
    else: return "60-0"
# --- 2. CONFIGURATIONS ---
IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/images"
MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/masks"
files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
categories = ["100-90", "90-85", "85-80", "80-75", "75-70", "70-60", "60-0"]
# VRAM PROTECTION: Separating numerical tracking from heavy image plotting
sayac = {cat: 0 for cat in categories}
cizim_icin_ornekler = {cat: [] for cat in categories}
# --- 3. ANALYSIS LOOP ---
model.eval()
print(f"🚀 Exhaustive analysis initiated for all {len(files)} testing files...")
with torch.no_grad():
    with torch.cuda.amp.autocast():
        for filename in tqdm(files):
            img_path = os.path.join(IMG_DIR, filename)
            mask_path = os.path.join(MASK_DIR, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if img is None or mask is None: continue
            input_tensor = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0).cuda() / 255.0
            output = model(input_tensor)
            if isinstance(output, (tuple, list)): output = output[0]
            pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()
            d_dice = calculate_dice(pred, mask, 1)
            c_dice = calculate_dice(pred, mask, 2)
            avg_dice = (d_dice + c_dice) / 2
            cat = get_cat(avg_dice)
            # Record every single instance
            sayac[cat] += 1
            # RAM PROTECTION: Cache only the first 3 examples of each class for plotting
            if len(cizim_icin_ornekler[cat]) < 3:
                cizim_icin_ornekler[cat].append({
                    'name': filename,
                    'img': img,
                    'pred': pred,
                    'mask': mask,
                    'avg': avg_dice
                })
# --- 4. RESULTS AND REPORTING ---
print("\n" + "="*40)
print("📊 MODEL PERFORMANCE AUDIT")
print("="*40)
total_processed = 0
for cat in categories:
    print(f"Category [{cat}]: {sayac[cat]} images")
    total_processed += sayac[cat]
print("="*40)
print(f"Validation verification: Total verified instances = {total_processed}")
print("\n🖼️ Rendering up to 3 exemplary outputs per accuracy bracket:")
for cat in categories:
    samples = cizim_icin_ornekler[cat]
    if len(samples) == 0: continue
    print(f"\n--- Category: {cat} (Total cohort count: {sayac[cat]}) ---")
    for b in samples:
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(b['img'], cmap='gray')
        plt.title("Original Fundus")
        plt.axis('off')
        plt.subplot(1, 3, 2)
        plt.imshow(b['mask'], cmap='jet')
        plt.title("Ground Truth (GT)")
        plt.axis('off')
        plt.subplot(1, 3, 3)
        plt.imshow(b['pred'], cmap='jet')
        plt.title(f"Prediction (Dice: {b['avg']:.2f})")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

Monte Carlo (MC) Dropout Advanced Interpretability

In [ ]:
import os, cv2, torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
def calculate_dice(pred, mask, label):
    pred_bin = (pred == label).astype(np.float32)
    mask_bin = (mask == label).astype(np.float32)
    intersection = np.sum(pred_bin * mask_bin)
    return (2.0 * intersection) / (np.sum(pred_bin) + np.sum(mask_bin) + 1e-7)
def enable_mc_dropout(model):
    """Enables all Dropout layers during inference."""
    for m in model.modules():
        if 'Drop' in m.__class__.__name__:
            m.train()
def get_mc_predictions(model, image_tensor, num_passes=20):
    """N stochastic passes to extract Mean Prediction, Heatmap, and Confidence Score."""
    model.eval()
    enable_mc_dropout(model)
    mc_preds = []
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            for _ in range(num_passes):
                output = model(image_tensor)
                if isinstance(output, (tuple, list)): output = output[0]
                mc_preds.append(F.softmax(output, dim=1))
    stacked_preds = torch.stack(mc_preds)
    mean_probs = torch.mean(stacked_preds, dim=0)
    final_pred = torch.argmax(mean_probs, dim=1).squeeze().cpu().numpy()
    max_probs = torch.max(mean_probs, dim=1)[0].squeeze().cpu().numpy()
    confidence_score = np.mean(max_probs) * 100
    variance = torch.var(stacked_preds, dim=0)
    uncertainty_map = torch.sum(variance, dim=1).squeeze().cpu().numpy()
    if uncertainty_map.max() > 0:
        uncertainty_map = (uncertainty_map - uncertainty_map.min()) / (uncertainty_map.max() - uncertainty_map.min())
    return final_pred, uncertainty_map, confidence_score
# Paths
IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/images"
MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/test_data/test_withmask/masks"
files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
target_categories = {
    "Flawless Cases (Dice: 95-100)": [],
    "Excellent Cases (Dice: 90-95)": [],
    "Challenging Cases (Dice: 60-80)": []
}
sample_limit = 3
model.eval()
print("🚀 Diagnostic Loop: Hunting for specified clinical thresholds...")
for filename in tqdm(files):
    if all(len(samples) >= sample_limit for samples in target_categories.values()):
        break
    img_path = os.path.join(IMG_DIR, filename)
    mask_path = os.path.join(MASK_DIR, filename)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: continue
    input_tensor = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0).cuda() / 255.0
    # Fast initial prediction filter
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            output = model(input_tensor)
            if isinstance(output, (tuple, list)): output = output[0]
            quick_pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()
    d_dice = calculate_dice(quick_pred, mask, 1)
    c_dice = calculate_dice(quick_pred, mask, 2)
    avg_dice = (d_dice + c_dice) / 2
    percentage = avg_dice * 100
    category_name = None
    if 95 <= percentage <= 100 and len(target_categories["Flawless Cases (Dice: 95-100)"]) < sample_limit:
        category_name = "Flawless Cases (Dice: 95-100)"
    elif 90 <= percentage < 95 and len(target_categories["Excellent Cases (Dice: 90-95)"]) < sample_limit:
        category_name = "Excellent Cases (Dice: 90-95)"
    elif 60 <= percentage < 80 and len(target_categories["Challenging Cases (Dice: 60-80)"]) < sample_limit:
        category_name = "Challenging Cases (Dice: 60-80)"
    if category_name:
        mc_pred, uncertainty, confidence_score = get_mc_predictions(model, input_tensor, num_passes=20)
        mc_d_dice = calculate_dice(mc_pred, mask, 1)
        mc_c_dice = calculate_dice(mc_pred, mask, 2)
        mc_avg_dice = (mc_d_dice + mc_c_dice) / 2
        target_categories[category_name].append({
            'isim': filename,
            'img': img,
            'mask': mask,
            'pred': mc_pred,
            'uncertainty': uncertainty,
            'dice_skoru': mc_avg_dice * 100,
            'guven_skoru': confidence_score
        })
print("\n" + "="*55)
print("🎨 MC-DROPOUT: SUCCESS AND CONFIDENCE MAPPING")
print("="*55)
for kat_adi, ornekler in target_categories.items():
    if len(ornekler) == 0: continue
    print(f"\n📌 {kat_adi}")
    for ornek in ornekler:
        plt.figure(figsize=(16, 4))
        plt.subplot(1, 4, 1)
        plt.imshow(ornek['img'], cmap='gray')
        plt.title(f"Original Fundus\n({ornek['isim']})")
        plt.axis('off')
        plt.subplot(1, 4, 2)
        plt.imshow(ornek['mask'], cmap='jet')
        plt.title("Doctor Ground Truth")
        plt.axis('off')
        plt.subplot(1, 4, 3)
        plt.imshow(ornek['pred'], cmap='jet')
        plt.title(f"Model Prediction\nAccuracy (Dice): {ornek['dice_skoru']:.1f}%")
        plt.axis('off')
        plt.subplot(1, 4, 4)
        plt.imshow(ornek['uncertainty'], cmap='hot')
        plt.title(f"Uncertainty Heatmap\nConfidence Score: {ornek['guven_skoru']:.1f}%")
        plt.colorbar(fraction=0.046, pad=0.04)
        plt.axis('off')
        plt.tight_layout()
        plt.show()


Extract & Analyze Perfect Dice Coordinates

In [ ]:

import os
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
ZIP_PATH = "/content/drive/MyDrive/yeni_glokom_proje/test_data/TEST.zip"
if not os.path.exists(ZIP_PATH):
    print(f"❌ ERROR: Zip location missing at {ZIP_PATH}")
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        all_files = z.namelist()
        raw_images = [f for f in all_files if f.startswith("images/") and f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        mask_files = [f for f in all_files if f.startswith("masks/") and f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        mask_dict = {os.path.splitext(os.path.basename(m))[0].lower(): m for m in mask_files}
        valid_images = []
        for img_path in raw_images:
            img_id = os.path.splitext(os.path.basename(img_path))[0].lower()
            if img_id in mask_dict:
                valid_images.append((img_path, mask_dict[img_id]))
        if len(valid_images) == 0:
            print("❌ ERROR: No matching Image/Mask pairs discovered!")
        else:
            chosen_img_path, chosen_mask_path = random.choice(valid_images)
            img_raw = cv2.imdecode(np.frombuffer(z.read(chosen_img_path), np.uint8), cv2.IMREAD_COLOR)
            img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
            mask_raw = cv2.imdecode(np.frombuffer(z.read(chosen_mask_path), np.uint8), cv2.IMREAD_GRAYSCALE)
            h, w, _ = img_raw.shape
            pts = np.argwhere(mask_raw > 0)
            if len(pts) == 0:
                print("⚠️ Warning: Mask parameters void. Execute cell again.")
            else:
                cy, cx = int(np.mean(pts[:, 0])), int(np.mean(pts[:, 1]))
                ymin_m, xmin_m = np.min(pts, axis=0)
                ymax_m, xmax_m = np.max(pts, axis=0)
                max_disk_side = max(xmax_m - xmin_m, ymax_m - ymin_m)
                # STRATEGY 1: STRICT 15% ZOOM-IN PADDING
                pad_15 = int(max_disk_side * 0.15)
                mavi_ymin, mavi_ymax = max(0, cy - (max_disk_side//2) - pad_15), min(h, cy + (max_disk_side//2) + pad_15)
                mavi_xmin, mavi_xmax = max(0, cx - (max_disk_side//2) - pad_15), min(w, cx + (max_disk_side//2) + pad_15)
                crop_mavi = img_rgb[mavi_ymin:mavi_ymax, mavi_xmin:mavi_xmax]
                hm, wm = crop_mavi.shape[:2]
                scale_m = 512 / max(hm, wm)
                nwm, nhm = int(wm * scale_m), int(hm * scale_m)
                resized_mavi = cv2.resize(crop_mavi, (nwm, nhm), interpolation=cv2.INTER_LINEAR)
                final_mavi = np.zeros((512, 512, 3), dtype=np.uint8)
                final_mavi[(512-nhm)//2:(512-nhm)//2+nhm, (512-nwm)//2:(512-nwm)//2+nwm] = resized_mavi
                # STRATEGY 2: CONSERVATIVE 40% WIDE FOV ZOOM-OUT PADDING
                pad_40 = int(max_disk_side * 0.40)
                yesil_ymin, yesil_ymax = max(0, cy - (max_disk_side//2) - pad_40), min(h, cy + (max_disk_side//2) + pad_40)
                yesil_xmin, yesil_xmax = max(0, cx - (max_disk_side//2) - pad_40), min(w, cx + (max_disk_side//2) + pad_40)
                crop_yesil = img_rgb[yesil_ymin:yesil_ymax, yesil_xmin:yesil_xmax]
                hy, wy = crop_yesil.shape[:2]
                scale_y = 512 / max(hy, wy)
                nwy, nhy = int(wy * scale_y), int(hy * scale_y)
                resized_yesil = cv2.resize(crop_yesil, (nwy, nhy), interpolation=cv2.INTER_LINEAR)
                final_yesil = np.zeros((512, 512, 3), dtype=np.uint8)
                final_yesil[(512-nhy)//2:(512-nhy)//2+nhy, (512-nwy)//2:(512-nwy)//2+nwy] = resized_yesil
                # VISUALIZATION PANEL
                fig, axes = plt.subplots(1, 3, figsize=(18, 6))
                axes[0].imshow(img_rgb)
                rect_m = plt.Rectangle((mavi_xmin, mavi_ymin), mavi_xmax-mavi_xmin, mavi_ymax-mavi_ymin, fill=False, color='cyan', linewidth=2, linestyle='--')
                rect_y = plt.Rectangle((yesil_xmin, yesil_ymin), yesil_xmax-yesil_xmin, yesil_ymax-yesil_ymin, fill=False, color='lime', linewidth=3)
                axes[0].add_patch(rect_m)
                axes[0].add_patch(rect_y)
                axes[0].set_title(f"1. Raw FOV Overlay\nFile: {os.path.basename(chosen_img_path)}\nDashed Cyan: Current | Solid Green: Proposed")
                axes[0].axis('off')
                axes[1].imshow(final_mavi)
                axes[1].set_title("2. Current 15% Crop Strategy (512x512)\nDisc bleeds out of frame bounds!")
                axes[1].axis('off')
                axes[2].imshow(final_yesil)
                axes[2].set_title("3. Proposed 40% Wide Crop (512x512)\nFull structural containment achieved.")
                axes[2].axis('off')
                plt.tight_layout()
                plt.show()
# Assuming results_dict exists in the global environment
try:
    perfect_dice_count = sum(len([d for d in results_dict[cat] if d >= 0.999]) for cat in results_dict)
    print(f"🎯 FLAWLESS PREDICTIONS (Dice=1): {perfect_dice_count} identified.")
except NameError:
    pass

YOLO Fallback Diagnostics & Testing

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import random
import os
import numpy as np
# 1. Drive Connection Validation
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
# 2. Locate Archived Checkpoint
model_path = '/content/drive/MyDrive/glokom_proje/yolo_disk_dedektor_yeni.pt'
if os.path.exists(model_path):
    yolo_disk_dedektor = YOLO(model_path)
    print("✅ Legacy Backup Model acquired securely from cloud infrastructure.")
else:
    print("❌ ERROR: Model path offline. Check designated tracking folder.")
test_images_path = "/content/yolo_env/images"
if os.path.exists(test_images_path) and len(os.listdir(test_images_path)) > 0:
    all_test_imgs = [os.path.join(test_images_path, f) for f in os.listdir(test_images_path)]
    random_test_imgs = random.sample(all_test_imgs, 2)
    # Visualization Layer
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    for i, img_path in enumerate(random_test_imgs):
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        results = yolo_disk_dedektor.predict(img, conf=0.5, verbose=False)
        if len(results[0].boxes) > 0:
            box = results[0].boxes.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            # Strict 20% validation padding
            pad_w, pad_h = int((x2 - x1) * 0.20), int((y2 - y1) * 0.20)
            x1_p, y1_p = max(0, x1 - pad_w), max(0, y1 - pad_h)
            x2_p, y2_p = min(w, x2 + pad_w), min(h, y2 + pad_h)
            res_plotted = results[0].plot()
            res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
            axes[0, i].imshow(res_rgb)
            axes[0, i].set_title(f"Backup Model Target Identification")
            crop = img_rgb[y1_p:y2_p, x1_p:x2_p]
            axes[1, i].imshow(cv2.resize(crop, (512, 512)))
            axes[1, i].set_title(f"MiT-B3 Normalized Input Stream")
        for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("❌ ERROR: Extraction failed. Diagnostic path `/content/yolo_env` offline.")

Advanced YOLO Output Auditing (Multi-Mask System)

In [ ]:
import os
import cv2
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
YOLO_INPUT_IMG = "/content/YOLO_Ile_Kirpilacaklar/images"
YOLO_INPUT_MASK = "/content/YOLO_Ile_Kirpilacaklar/masks"
print("🔍 ======================================================= 🔍")
print("🩻 HIGH-PRECISION MULTI-MASK TARGETING AUDIT")
print("🔍 ======================================================= 🔍\n")
def apply_glaucoma_filters(img):
    b, g, r = cv2.split(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g_clahe = clahe.apply(g)
    return cv2.equalizeHist(g_clahe)
def apply_padding(image, target_size=512):
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    x_off = (target_size - new_w) // 2
    y_off = (target_size - new_h) // 2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return canvas
butun_resimler = glob.glob(os.path.join(YOLO_INPUT_IMG, "*.*"))
refuge_samples = [f for f in butun_resimler if "refuge" in os.path.basename(f).lower() or "g0" in os.path.basename(f).lower() or "n0" in os.path.basename(f).lower() or "t0" in os.path.basename(f).lower() or "v0" in os.path.basename(f).lower()]
if len(refuge_samples) == 0:
    print("❌ CRITICAL: Directory offline! Execute prerequisite extraction pipeline.")
else:
    secilen_ornekler = random.sample(refuge_samples, min(4, len(refuge_samples)))
    print(f"🚀 Randomly acquired {len(secilen_ornekler)} REFUGE2 validation parameters. Processing...\n")
    for idx, img_path in enumerate(secilen_ornekler):
        filename = os.path.basename(img_path)
        mask_path = os.path.join(YOLO_INPUT_MASK, filename)
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            print(f"⚠️ Warning: Checkpoint [{filename}] corrupted, skipping.")
            continue
        h, w, _ = img.shape
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pts = np.argwhere(mask > 0)
        if len(pts) > 0:
            y_coords = pts[:, 0]
            x_coords = pts[:, 1]
            center_y = int(np.mean(y_coords))
            center_x = int(np.mean(x_coords))
            # Fixed Surgical Field of View (450x450px)
            crop_dim = 450
            y_min = max(0, center_y - (crop_dim // 2))
            y_max = min(h, center_y + (crop_dim // 2))
            x_min = max(0, center_x - (crop_dim // 2))
            x_max = min(w, center_x + (crop_dim // 2))
            durum = "✅ Mask-Centric Precision Alignment Validated"
        else:
            # Fallback Center Constraint
            crop_size_h, crop_size_w = int(h * 0.25), int(w * 0.25)
            y_min, y_max = (h // 2) - (crop_size_h // 2), (h // 2) + (crop_size_h // 2)
            x_min, x_max = (w // 2) - (crop_size_w // 2), (w // 2) + (crop_size_w // 2)
            durum = "⚠️ Warning: Zero-pixel mask. Static center fallback engaged."
        img_crop = img[y_min:y_max, x_min:x_max]
        img_filtered = apply_glaucoma_filters(img_crop)
        img_sota = apply_padding(img_filtered, 512)
        print(f"📝 [SAMPLE {idx+1}] File: {filename} ({w}x{h} px)")
        print(f"🎯 Status: {durum} | Cropped Resolution: {img_crop.shape[1]}x{img_crop.shape[0]} px")
        print("-" * 70)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(img_rgb)
        rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=False, color='yellow', linewidth=3)
        axes[0].add_patch(rect)
        axes[0].set_title("Raw Source Geometry")
        axes[0].axis('off')
        axes[1].imshow(img_sota, cmap='gray')
        axes[1].set_title("512x512 SOTA Filtered Output")
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
print("\n🔍 Clinical Inspection routine completed. Verify focal alignment manually.")

# Sumamry


Mount Drive

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

In [ ]:
# Extract the ZIP file
!rm -rf /content/data  # remove the old directory
!unzip -q "/content/drive/MyDrive/Dataset/mix_ref_dris_data.zip" -d /content/data

print("✅ Data loaded successfully!")
print("Total Images:", len(os.listdir("/content/data/images")))

In [ ]:
model = smp.Unet(
    encoder_name="mit_b3",
    encoder_weights="imagenet",
    in_channels=1,
    classes=3,
).cuda()
print("🚀 1-Channel MiT-B3 Model ready on GPU!")

In [ ]:
from torch import nn
# Loss Functions
criterion_ce = nn.CrossEntropyLoss()
criterion_dice = smp.losses.DiceLoss(mode='multiclass')
# Optimizer configuration (Increased LR and Weight Decay added)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
def calculate_loss(pred, target):
    return 0.5 * criterion_ce(pred, target) + 0.5 * criterion_dice(pred, target)
print("✅ Hybrid Loss system and Optimizer initialized!")

In [ ]:
import os, cv2, shutil, numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
# 1. CLEANUP AND SETUP
DRIVE_IMG = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
DRIVE_MASK = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
LOCAL_IMG = "/content/new_train_data/raw_cropped"
LOCAL_MASK = "/content/new_train_data/masks"
# Reset directories
for path in [DRIVE_IMG, DRIVE_MASK, LOCAL_IMG, LOCAL_MASK]:
    if os.path.exists(path): shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
yolo_model = YOLO('/content/drive/MyDrive/yeni_glokom_proje/yolo_save/yolo_disk_dedektor_yeniv2.pt')
IMAGES_DIR = "/content/YOLO_Ile_Kirpilacaklar/images"
MASKS_DIR = "/content/YOLO_Ile_Kirpilacaklar/masks"
files = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"🚀 Scanning {len(files)} images, processing those with valid masks...")
# 2. PROCESSING LOOP
for idx, filename in enumerate(tqdm(files)):
    mask_path = os.path.join(MASKS_DIR, filename)
    if not os.path.exists(mask_path): continue
    img = cv2.imread(os.path.join(IMAGES_DIR, filename))
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None: continue
    h, w = img.shape[:2]
    is_refuge = "refuge" in filename.lower() or max(h, w) > 1200
    if is_refuge:
        pts = np.argwhere(mask > 0)
        if len(pts) > 0:
            cy, cx = int(np.mean(pts[:, 0])), int(np.mean(pts[:, 1]))
            y_min, y_max = max(0, cy - 225), min(h, cy + 225)
            x_min, x_max = max(0, cx - 225), min(w, cx + 225)
        else:
            y_min, y_max, x_min, x_max = h//2-250, h//2+250, w//2-250, w//2+250
    else:
        results = yolo_model.predict(img, conf=0.4, verbose=False)
        if len(results[0].boxes) > 0:
            x1, y1, x2, y2 = map(int, results[0].boxes.xyxy[0])
            pad = int((x2 - x1) * 0.15)
            y_min, y_max = max(0, y1 - pad), min(h, y2 + pad)
            x_min, x_max = max(0, x1 - pad), min(w, x2 + pad)
        else:
            y_min, y_max, x_min, x_max = h//2-250, h//2+250, w//2-250, w//2+250
    crop = img[y_min:y_max, x_min:x_max]
    mask_crop = mask[y_min:y_max, x_min:x_max]
    # Save to local and Drive simultaneously
    cv2.imwrite(os.path.join(LOCAL_IMG, filename), crop)
    cv2.imwrite(os.path.join(LOCAL_MASK, filename), mask_crop)
    cv2.imwrite(os.path.join(DRIVE_IMG, filename), crop)
    cv2.imwrite(os.path.join(DRIVE_MASK, filename), mask_crop)
    if idx % 100 == 0:
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); plt.title("Raw Crop"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(mask_crop, cmap='gray'); plt.title("Raw Mask"); plt.axis('off')
        plt.show()
print(f"✅ DONE! A total of {len(os.listdir(DRIVE_IMG))} cleaned images and masks are saved to Drive.")

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
INPUT_DIR = "/content/new_train_data/raw_cropped"
FINAL_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
os.makedirs(FINAL_DIR, exist_ok=True)
def apply_glaucoma_filters(img):
    # 1. Extract Green Channel (Provides maximum contrast for Disc and Cup)
    if len(img.shape) == 3:
        b, g, r = cv2.split(img)
        img = g
    # 2. CLAHE (Local contrast enhancement - Critical for disc boundaries)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(img)
    # 3. Global Histogram Equalization (Brightness balance)
    return cv2.equalizeHist(clahe_img)
def pad_to_512(img):
    # Fit into 512x512 canvas with black bars instead of stretching
    h, w = img.shape[:2]
    scale = 512 / max(h, w)
    resized = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((512, 512), dtype=np.uint8)
    y_off = (512 - resized.shape[0]) // 2
    x_off = (512 - resized.shape[1]) // 2
    canvas[y_off:y_off+resized.shape[0], x_off:x_off+resized.shape[1]] = resized
    return canvas
print("🎨 Applying filters and securing 512x512 aspect ratio...")
files = os.listdir(INPUT_DIR)
for file_name in tqdm(files):
    img = cv2.imread(os.path.join(INPUT_DIR, file_name))
    if img is None: continue
    # Apply filters
    filtered_img = apply_glaucoma_filters(img)
    # Apply padding
    final_img = pad_to_512(filtered_img)
    # Save
    cv2.imwrite(os.path.join(FINAL_DIR, file_name), final_img)
print(f"\n✅ PROCESS COMPLETE! {len(files)} filtered 512x512 images saved to Drive.")

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Grayscale (Tek Kanal) SOTA verisine özel, maske korumalı kararlı transform yapısı
train_transform = A.Compose([
    # 1. Geometrik (Açı ve konum)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.5),

    # 2. SOTA ÖZELLİKLER: Biyolojik retina esnemeleri ve bükülmeleri
    # Maskelerin (0,1,2) bozulmaması için mask_value interpolasyonunu sabitledik
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    A.GridDistortion(p=0.5),

    # 3. Işık ve Stil (Patlamaları/karanlıkları engeller)
    # 🔥 HATA VERDİREN HueSaturationValue BURADAN TAMAMEN KALDIRILDI!
    # Grayscale resimde renk tonu olmayacağı için sadece parlaklık/kontrast kalmalı.
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),

    # 4. Keskinlik ve Gürültü (Filtre dozajlarına bağışıklık)
    A.Sharpen(alpha=(0.2, 0.4), p=0.3),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),

    # 5. ÇIKIŞ (MiT-B3 eğitimi için tek kanala indirgenmiş normalizasyon hattı)
    A.Normalize(mean=(0.485,), std=(0.229,)),
    ToTensorV2()
])

print("✅ Sinsi hata yapısı kaldırıldı, Grayscale SOTA transformu hazır reis!")

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
# YOLO Fine-Tuning constraints
train_transform_yolo = A.Compose([
    # 2. LIGHTING AND COLOR (Simulate sensor variance across devices)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    # 3. SHARPNESS AND NOISE (Simulate camera/lens degradation)
    A.Sharpen(alpha=(0.2, 0.4), p=0.3),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
    # 4. OUTPUT (Normalization removed, ToTensorV2 retained)
    # YOLO handles internal 0-1 normalization natively.
    ToTensorV2()
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
# ==============================================================================
# 🛡️ STEP 1: DIMENSIONAL-GUARANTEE DATASET CLASS
# ==============================================================================
class GlaucomaGrayscaleDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            return torch.zeros((1, 512, 512), dtype=torch.float32), torch.zeros((512, 512), dtype=torch.long)
        # 🚨 DIMENSION SYNC PROTOCOL (Prevents runtime crash)
        if img.shape != mask.shape:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask[mask > 2] = 0 # Out-of-bounds artifact cleaning
        img = np.expand_dims(img, axis=-1)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask)
        return img, mask.long()
# ==============================================================================
# 🚀 STEP 2: TURBO DATALOADER
# ==============================================================================
TRAIN_IMG_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/images"
TRAIN_MASK_DIR = "/content/drive/MyDrive/yeni_glokom_proje/new_train_data/masks"
train_ds = GlaucomaGrayscaleDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
# ==============================================================================
# 🔥 STEP 3: TRAINING LOOP
# ==============================================================================
torch.cuda.empty_cache()
num_epochs = 30
model.train()
print("🚀 TRAINING INITIATED... A100 FULL THROTTLE!")
for epoch in range(num_epochs):
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    for images, masks in loop:
        images = images.cuda()
        masks = masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        save_path = f"/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_v8_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Checkpoint secured: {save_path}")
final_path = "/content/drive/MyDrive/yeni_glokom_proje/mit-b3_save/glaucoma_mitb3_v8.pth"
torch.save(model.state_dict(), final_path)
print(f"\n🎉 OPERATION COMPLETE! Final model located at: {final_path}")